In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
train = pd.read_csv("/train.csv")
test = pd.read_csv("/test.csv")

train_id = train["Id"]
test_id = test["Id"]

train.drop("Id", axis=1, inplace=True)
test.drop("Id", axis=1, inplace=True)

train.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train = train.drop(
    train[(train['GrLivArea'] > 4000) &
          (train['SalePrice'] < 300000)].index
)

In [ ]:
y = np.log1p(train["SalePrice"])
train.drop("SalePrice", axis=1, inplace=True)

In [ ]:
all_data = pd.concat([train, test]).reset_index(drop=True)
print("Combined shape:", all_data.shape)

Combined shape: (2917, 79)


In [ ]:
cols = ['MSSubClass', 'OverallCond', 'YrSold', 'MoSold']
for col in cols:
    all_data[col] = all_data[col].astype(str)

In [ ]:
for col in all_data.select_dtypes(include=['object']).columns:
    all_data[col] = all_data[col].fillna("None")

for col in all_data.select_dtypes(exclude=['object']).columns:
    all_data[col] = all_data[col].fillna(all_data[col].median())

print("Total missing:", all_data.isnull().sum().sum())

Total missing: 0


In [ ]:
all_data['TotalSF'] = (
    all_data['TotalBsmtSF'] +
    all_data['1stFlrSF'] +
    all_data['2ndFlrSF']
)

all_data['TotalBath'] = (
    all_data['FullBath'] +
    0.5 * all_data['HalfBath'] +
    all_data['BsmtFullBath'] +
    0.5 * all_data['BsmtHalfBath']
)

all_data['HouseAge'] = (
    all_data['YrSold'].astype(int) -
    all_data['YearBuilt']
)

In [ ]:
numeric_feats = all_data.select_dtypes(exclude=['object']).columns

skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x))
skewed_feats = skewed_feats[skewed_feats > 0.75].index

for feat in skewed_feats:
    all_data[feat] = np.log1p(all_data[feat])

print("Skew corrected features:", len(skewed_feats))

Skew corrected features: 20


In [ ]:
all_data = pd.get_dummies(all_data)
print("After encoding:", all_data.shape)

After encoding: (2917, 350)


In [ ]:
X_train = all_data[:len(y)]
X_test = all_data[len(y):]

print(X_train.shape, X_test.shape)

(1458, 350) (1459, 350)


In [ ]:
ridge = Ridge(alpha=15)

score = np.sqrt(-cross_val_score(
    ridge, X_train, y,
    scoring="neg_mean_squared_error",
    cv=5
))

print("Ridge CV RMSE:", score.mean())

Ridge CV RMSE: 0.11485325817667258


In [ ]:
lasso = Lasso(alpha=0.0005)

score_lasso = np.sqrt(-cross_val_score(
    lasso, X_train, y,
    scoring="neg_mean_squared_error",
    cv=5
))

print("Lasso CV RMSE:", score_lasso.mean())

Lasso CV RMSE: 0.11096685318411119


In [ ]:
ridge.fit(X_train, y)
lasso.fit(X_train, y)

pred_ridge = np.expm1(ridge.predict(X_test))
pred_lasso = np.expm1(lasso.predict(X_test))

final_pred = (pred_ridge + pred_lasso) / 2

In [22]:
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": final_pred
})

submission.to_csv("submission.csv", index=False)
submission.head()

,Id,SalePrice
0,1461,122957.946572
1,1462,156214.548648
2,1463,182474.766280
3,1464,201953.287608
4,1465,196203.153257
